In [70]:
import json
from typing import Dict, Any, List

In [71]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/jupyter-tutorial/hf_models/Qwen3-4B", padding_side='left')

def count_tokens_a(text: str) -> int:
    """Count the number of tokens in the text using the agent's tokenizer"""
    return len(tokenizer.encode(text, add_special_tokens=False))

def filter_answers(ans: List[str|Dict[str, str]]) -> List[Dict[str, str]]:
    r"""Filter answers to ensure they are in the correct format"""
    def basic_checks(a1: Dict[str, str])->bool:
        # check required keys
        required_keys = ['answer']
        if all((key in a1) and isinstance(a1[key], str) for key in required_keys):
            if len(a1['answer']) == 1 and (a1['answer'] not in 'ABCDabcd'):
                    return False
            check_len = count_tokens_a(a1['answer'])
            if check_len < 50:
                check_len += count_tokens_a(a1.get('reasoning', 'None'))
                if check_len < 512:
                    # check answer format - EXTRA checks
                    # if len(a1['answer']) == 1 and a1['answer'].upper() in 'ABCD':
                    return True
        return False

    filtered_answers = []
    for i, a in enumerate(ans):
        if isinstance(a, dict):
            if basic_checks(a):
                filtered_answers.append(a)
            else:
                filtered_answers.append(None)
        elif isinstance(a, str):
            # Basic checks: at least with correct JSON format
            try:
                a1 = json.loads(a)
                if basic_checks(a1):
                    filtered_answers.append(a1)
                else:
                    filtered_answers.append(None)
            except json.JSONDecodeError:
                # If JSON decoding fails, skip this answer
                print(f"Skipping invalid JSON at index {i}: {a}")
                filtered_answers.append(None)
                continue
        else:
            # If the answer is neither a dict nor a str, skip it
            print(f"Skipping unsupported type at index {i}: {type(a)}")
            filtered_answers.append(None)
    return filtered_answers

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/jupyter-tutorial/hf_models/Qwen3-4B", padding_side='left')

def count_tokens_q(text: str) -> int:
    """Count the number of tokens using Qwen3-4B tokenizer"""
    return len(tokenizer.encode(text, add_special_tokens=False))

def filter_questions(questions: List[str|Dict[str, str|Any]]) -> List[Dict[str, str|Any]]:
    def basic_checks(q2: Dict[str, str])->bool:
        # check required keys
        required_keys = ['topic', 'question', 'choices', 'answer']
        if all((key in q2) for key in required_keys):
            # check choices format
            checks = all(isinstance(choice, str) and len(choice) > 2 and choice[0].upper() in 'ABCD' for choice in q2['choices'])
            if isinstance(q2['choices'], list) and len(q2['choices']) == 4 and checks:
                # check answer format
                # Check token length
                check_len = sum(count_tokens_q(q2[k]) for k in ['question', 'answer'])
                check_len += sum(count_tokens_q(choice) for choice in q2['choices']) - 15
                if check_len < 130:
                    if check_len + count_tokens_q(q2.get('explanation', 'None')) <= 1024:
                        # Extra Checks: (PLUS checks) len(q2['answer']) == 1 and q2['answer'].upper() in 'ABCD':
                        if isinstance(q2['answer'], str):
                            return True
        return False
    correct_format_question = []
    for i, q in enumerate(questions):
        if isinstance(q, dict):
            if basic_checks(q):
                correct_format_question.append(q)
        elif isinstance(q, str):
            try:
                q1 = json.loads(q)
                if basic_checks(q1):
                    correct_format_question.append(q1)
            except json.JSONDecodeError:
                # If JSON decoding fails, skip this answer
                print(f"Skipping invalid JSON at index {i}: {q}")
                continue
        else:
            continue
    if len(correct_format_question) >= 0.5 * len(questions):
        return correct_format_question
    return list()

# Execute Q AGENT

In [72]:
!python -m agents.question_agent \
    --model_name /jupyter-tutorial/AAIPL_129_212_191_46/ckpt/ckpt_q_agent/ \
    --output_file "outputs/questions.json" \
    --num_questions 50 \
    --verbose

STEPS: 100%|████████████████████████████████████| 10/10 [03:48<00:00, 22.81s/it]
Generated 50 questions!
{
  "topic": "Puzzles/Seating Arrangements (Linear, Circular)",
  "question": "In a circular arrangement of seven people, P is sitting between Q and R, and S is sitting immediate right of T. U is sitting immediate left of Q. Who is sitting immediate right of P?",
  "choices": ["A) Q", "B) R", "C) S", "D) T"],
  "answer": "D",
  "explanation": "Since P is sitting between Q and R, and S is sitting immediate right of T, the order must be U, Q, P, T, S, R, Q. Therefore, T is sitting immediate right of P." 
}
{
  "topic": "Blood Relations and Family Tree/Puzzles involving generations and family tree logic",
  "question": "Pointing to a photograph, Rohan says, 'He is the son of the brother of my mother.' How is the man in the photograph related to Rohan?",
  "choices": ["A) Father", "B) Uncle", "C) Cousin", "D) Brother"],
  "answer": "D",
  "explanation": "The man is the son of the brothe

In [73]:

with open("outputs/questions.json", "r") as f:
    questions = json.load(f)

filtered_questions = filter_questions(questions)

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Further filtering will happen with our Oracle (not shown here) which also have its own answer for the question.
# If Q-agent answer to its own question is wrong, then that question will not be considered.
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

with open("outputs/filtered_questions.json", "w") as f:
    json.dump(filtered_questions, f, indent=4)

len(filtered_questions)

38

# Execute A AGENT

In [78]:
!python -m agents.answer_agent \
    --model_name /jupyter-tutorial/AAIPL_129_212_191_46/ckpt/ckpt_q_agent/ \
    --input_file "outputs/filtered_questions.json" \
    --output_file "outputs/answers.json" \
    --verbose

Loading checkpoint shards: 100%|██████████████████| 3/3 [00:02<00:00,  1.07it/s]
STEPS: : 9batch [00:45,  5.02s/batch]                                           

=== Question 1 ===
Question: In a circular arrangement of seven people, P is sitting between Q and R, and S is sitting immediate right of T. U is sitting immediate left of Q. Who is sitting immediate right of P?
Expected: D
Model Answer:
{
  "answer": "B",
  "reasoning": "Since U is sitting immediate left of Q, and P is sitting between Q and R, R is sitting immediate right of P."}

=== Question 2 ===
Question: Pointing to a photograph, Rohan says, 'He is the son of the brother of my mother.' How is the man in the photograph related to Rohan?
Expected: D
Model Answer:
{
  "answer": "A",
  "reasoning": "The man in the photograph is the son of the brother of Rohan's mother, making him the son of Rohan's uncle. Therefore, the man is Rohan's father."}

=== Question 3 ===
Question: If A + B means A is the brother of B; A x B means 

In [79]:
with open("outputs/answers.json", "r") as f:
    answers = json.load(f)
filtered_answers = filter_answers(answers)
print(len(answers))
len(filtered_answers)

38


38

In [80]:
# calculate scores...
N = len(filtered_questions)
assert N == len(filtered_answers), "Number of questions and answers must match."
num_correct_answers = len([1 for q,a in zip(filtered_questions, filtered_answers) if a is not None and q['answer'] == a['answer']])

# Here the answer may be correct, but since q['answer'] is not an option letter is not there, we face problems
# Below shown is one way of simple string parsing
num_correct_answers = len([1 for q,a in zip(filtered_questions, filtered_answers) if a is not None and q['answer'][0] == a['answer']])

a_score = num_correct_answers*100/(N+1e-9)
q_score = (N-num_correct_answers)*100/(N+1e-9)
# Announce the scores
print(f"Number of questions: {N}")
print(f"Number of correct answers: {num_correct_answers}")
print("Scores:")
print(f"Team B: A-agent score: {a_score:.2f}")
print(f"Team A: Q-agent score: {q_score:.2f}")
print(f"Innings 1 winner: {'Team A' if q_score > a_score else 'Team B' if q_score < a_score else 'Draw'}")
# DRAW case is not HANDLED now

Number of questions: 38
Number of correct answers: 14
Scores:
Team B: A-agent score: 36.84
Team A: Q-agent score: 63.16
Innings 1 winner: Team A


In [ ]:
# !python -m agents.answer_agent \
#     --model_name /jupyter-tutorial/AAIPL_maha/ckpt_qAgent/ \
#     --input_file /jupyter-tutorial/AAIPL_maha/data/test-chota.json \
#     --output_file "outputs/answers.json" \
#     --verbose